# CT-RATE Radiology Report EDA

This notebook preprocesses CT-RATE radiology reports and structured labels, removes duplicate reports, and computes descriptive statistics suitable for publication.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'figure.dpi': 150,
})

DATA_DIR = Path('../data')
FIGURES_DIR = Path('../figures')
TABLES_DIR = Path('../tables')

## 1. Load Raw Data

In [ ]:
train_reports = pd.read_csv(DATA_DIR / 'train_reports.csv')
valid_reports = pd.read_csv(DATA_DIR / 'validation_reports.csv')
train_labels = pd.read_csv(DATA_DIR / 'train_predicted_labels.csv')
valid_labels = pd.read_csv(DATA_DIR / 'valid_predicted_labels.csv')

print(f'Train reports  : {len(train_reports):,} rows')
print(f'Valid reports  : {len(valid_reports):,} rows')
print(f'Train labels   : {len(train_labels):,} rows')
print(f'Valid labels   : {len(valid_labels):,} rows')

## 2. Merge Train / Validation and Attach Labels

In [ ]:
train_reports['split'] = 'train'
valid_reports['split'] = 'valid'

reports = pd.concat([train_reports, valid_reports], ignore_index=True)

labels = pd.concat([train_labels, valid_labels], ignore_index=True)

df = reports.merge(labels, on='VolumeName', how='inner')
print(f'After merge: {len(df):,} rows')
df.head(2)

## 3. Remove Duplicate Reports

Volumes sharing identical `Findings_EN` **and** `Impressions_EN` text are considered duplicates (e.g., repeated acquisitions of the same scan). We keep the first occurrence per unique (Findings, Impressions) pair and record provenance.

In [ ]:
# Normalise whitespace before comparing
for col in ['Findings_EN', 'Impressions_EN']:
    df[col] = df[col].astype(str).str.strip()

n_before = len(df)
df_dedup = df.drop_duplicates(subset=['Findings_EN', 'Impressions_EN'], keep='first').copy()
n_after = len(df_dedup)
n_removed = n_before - n_after

print(f'Rows before deduplication : {n_before:,}')
print(f'Rows after  deduplication : {n_after:,}')
print(f'Duplicates removed        : {n_removed:,} ({n_removed/n_before*100:.1f}%)')

# Split-level breakdown
print('\nPost-deduplication split distribution:')
print(df_dedup['split'].value_counts().to_string())

## 4. Save Preprocessed Data

In [ ]:
out_path = DATA_DIR / 'CT-RATE_reports.csv'
df_dedup.to_csv(out_path, index=False)
print(f'Saved: {out_path}  ({len(df_dedup):,} rows × {df_dedup.shape[1]} columns)')

## 5. Text Statistics for Findings and Impressions

Word counts are computed by whitespace tokenisation. Character counts exclude leading/trailing whitespace.

In [ ]:
def word_count(s):
    return len(str(s).split())

def char_count(s):
    return len(str(s).strip())

for section in ['Findings_EN', 'Impressions_EN']:
    df_dedup[f'{section}_words'] = df_dedup[section].apply(word_count)
    df_dedup[f'{section}_chars'] = df_dedup[section].apply(char_count)

df_dedup.head(2)

In [ ]:
sections = ['Findings_EN', 'Impressions_EN']
stat_cols = ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']
stat_rename = {'count': 'N', 'mean': 'Mean', 'std': 'SD',
               'min': 'Min', '25%': 'Q1', '50%': 'Median',
               '75%': 'Q3', 'max': 'Max'}

rows = []
for subset_name, subset_df in [('Overall', df_dedup),
                                ('Train',   df_dedup[df_dedup['split'] == 'train']),
                                ('Valid',   df_dedup[df_dedup['split'] == 'valid'])]:
    for section in sections:
        col = f'{section}_words'
        desc = subset_df[col].describe()
        row = {'Subset': subset_name,
               'Section': section.replace('_EN', ''),
               'N': int(desc['count']),
               'Mean': round(desc['mean'], 1),
               'SD':   round(desc['std'],  1),
               'Min':  int(desc['min']),
               'Q1':   int(desc['25%']),
               'Median': int(desc['50%']),
               'Q3':   int(desc['75%']),
               'Max':  int(desc['max'])}
        rows.append(row)

word_stats = pd.DataFrame(rows)
print('Table 1. Word-count statistics for Findings and Impressions sections')
print(word_stats.to_string(index=False))

### Figure 1 – Word-count distributions

In [ ]:
split_palette = {'train': '#4878CF', 'valid': '#D65F5F'}
split_labels  = {'train': 'Train', 'valid': 'Validation'}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, section, title in zip(axes,
                               ['Findings_EN_words', 'Impressions_EN_words'],
                               ['Findings', 'Impressions']):
    for split, color in split_palette.items():
        subset = df_dedup[df_dedup['split'] == split][section]
        ax.hist(subset, bins=60, alpha=0.65, color=color,
                label=split_labels[split], edgecolor='none')
    ax.set_xlabel('Word count')
    ax.set_ylabel('Number of reports')
    ax.set_title(title)
    ax.legend()
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

fig.suptitle('Figure 1. Distribution of word counts in radiology reports',
             fontsize=12, y=1.02)
fig.tight_layout()
plt.savefig(FIGURES_DIR / 'fig1_word_count_distribution.png',
            bbox_inches='tight', dpi=200)
plt.show()

## 6. Structured Label EDA

### 6.1 Label prevalence (proportion of positive cases)

In [ ]:
label_cols = [c for c in train_labels.columns if c != 'VolumeName']
print(f'Number of label categories: {len(label_cols)}')
print(label_cols)

In [ ]:
def label_stats(subset_df, subset_name):
    counts = subset_df[label_cols].sum()
    prevalence = (subset_df[label_cols].mean() * 100).round(1)
    return pd.DataFrame({
        'Subset': subset_name,
        'N_positive': counts.values.astype(int),
        'Prevalence_%': prevalence.values
    }, index=label_cols)

overall_stats = label_stats(df_dedup, 'Overall')
train_stats   = label_stats(df_dedup[df_dedup['split'] == 'train'], 'Train')
valid_stats   = label_stats(df_dedup[df_dedup['split'] == 'valid'], 'Valid')

prevalence_table = pd.concat([overall_stats, train_stats, valid_stats], axis=0)
prevalence_wide = prevalence_table.pivot_table(
    index=prevalence_table.index,
    columns='Subset',
    values='Prevalence_%'
)[['Overall', 'Train', 'Valid']]

# Sort by overall prevalence
prevalence_wide = prevalence_wide.sort_values('Overall', ascending=False)

print('Table 2. Prevalence (%) of each label category')
print(prevalence_wide.to_string())

### Figure 2 – Label prevalence bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(prevalence_wide))
w = 0.28

ax.bar(x - w, prevalence_wide['Overall'], width=w, label='Overall',
       color='#555555', alpha=0.85)
ax.bar(x,     prevalence_wide['Train'],   width=w, label='Train',
       color='#4878CF', alpha=0.85)
ax.bar(x + w, prevalence_wide['Valid'],   width=w, label='Validation',
       color='#D65F5F', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(prevalence_wide.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Prevalence (%)')
ax.set_title('Figure 2. Prevalence of structured label categories')
ax.legend()
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'fig2_label_prevalence.png',
            bbox_inches='tight', dpi=200)
plt.show()

### 6.2 Number of positive labels per report

In [ ]:
df_dedup['n_labels'] = df_dedup[label_cols].sum(axis=1)

rows_nlabels = []
for subset_name, subset_df in [('Overall', df_dedup),
                                ('Train',   df_dedup[df_dedup['split'] == 'train']),
                                ('Valid',   df_dedup[df_dedup['split'] == 'valid'])]:
    d = subset_df['n_labels'].describe()
    rows_nlabels.append({
        'Subset': subset_name,
        'N': int(d['count']),
        'Mean': round(d['mean'], 2),
        'SD':   round(d['std'],  2),
        'Min':  int(d['min']),
        'Median': int(d['50%']),
        'Max':  int(d['max']),
    })

nlabels_stats = pd.DataFrame(rows_nlabels)
print('Table 3. Number of positive labels per report')
print(nlabels_stats.to_string(index=False))

### Figure 3 – Distribution of positive label count per report

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

max_labels = int(df_dedup['n_labels'].max())
bins = np.arange(-0.5, max_labels + 1.5, 1)

for split, color in split_palette.items():
    subset = df_dedup[df_dedup['split'] == split]['n_labels']
    ax.hist(subset, bins=bins, alpha=0.65, color=color,
            label=split_labels[split], edgecolor='white', linewidth=0.4)

ax.set_xlabel('Number of positive labels per report')
ax.set_ylabel('Number of reports')
ax.set_title('Figure 3. Distribution of positive label count per report')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'fig3_n_labels_distribution.png',
            bbox_inches='tight', dpi=200)
plt.show()

### 6.3 Label co-occurrence heatmap

In [ ]:
label_matrix = df_dedup[label_cols].astype(int)
cooccur = label_matrix.T.dot(label_matrix)

# Normalise to Jaccard similarity: |A∩B| / |A∪B|
n_pos = np.diag(cooccur.values)
denom = np.add.outer(n_pos, n_pos) - cooccur.values
jaccard = pd.DataFrame(
    np.where(denom == 0, 0, cooccur.values / denom),
    index=label_cols, columns=label_cols
)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.eye(len(label_cols), dtype=bool)

sns.heatmap(
    jaccard, mask=mask, cmap='Blues', vmin=0, vmax=0.5,
    linewidths=0.4, linecolor='white',
    annot=True, fmt='.2f', annot_kws={'size': 7},
    cbar_kws={'label': 'Jaccard similarity', 'shrink': 0.7},
    ax=ax
)
ax.set_title('Figure 4. Label co-occurrence (Jaccard similarity)', pad=10)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0,  labelsize=8)

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'fig4_label_cooccurrence.png',
            bbox_inches='tight', dpi=200)
plt.show()

## 7. Summary Table (Publication-ready)

In [ ]:
def summary_row(subset_df, name):
    n = len(subset_df)
    fw_med = int(subset_df['Findings_EN_words'].median())
    fw_iqr = (int(subset_df['Findings_EN_words'].quantile(0.25)),
              int(subset_df['Findings_EN_words'].quantile(0.75)))
    iw_med = int(subset_df['Impressions_EN_words'].median())
    iw_iqr = (int(subset_df['Impressions_EN_words'].quantile(0.25)),
              int(subset_df['Impressions_EN_words'].quantile(0.75)))
    nl_med = subset_df['n_labels'].median()
    nl_iqr = (subset_df['n_labels'].quantile(0.25),
              subset_df['n_labels'].quantile(0.75))
    return {
        'Subset': name,
        'N': n,
        'Findings words, median (IQR)': f"{fw_med} ({fw_iqr[0]}–{fw_iqr[1]})",
        'Impressions words, median (IQR)': f"{iw_med} ({iw_iqr[0]}–{iw_iqr[1]})",
        'Positive labels, median (IQR)': f"{nl_med:.0f} ({nl_iqr[0]:.0f}–{nl_iqr[1]:.0f})",
    }

summary = pd.DataFrame([
    summary_row(df_dedup, 'Overall'),
    summary_row(df_dedup[df_dedup['split'] == 'train'], 'Train'),
    summary_row(df_dedup[df_dedup['split'] == 'valid'], 'Validation'),
])

print('Table 4. Summary statistics of the CT-RATE dataset')
print(summary.to_string(index=False))

In [ ]:
print('\nFinal dataset shape:', df_dedup.shape)
print('Columns:', list(df_dedup.columns))